In [1]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_parquet
from pyspark.sql.functions import col, dayofweek, hour, when, round

# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [3]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi02")
zone_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\zone")

In [9]:
zone_df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [11]:
zones_pickup = zone_df \
    .withColumnRenamed("LocationID", "PU_LocationID") \
    .withColumnRenamed("Zone", "PU_zone") \
    .withColumnRenamed("Borough", "PU_borough") \
    .withColumnRenamed("service_zone", "PU_service_zone")


zones_dropoff = zone_df \
    .withColumnRenamed("LocationID", "DO_LocationID") \
    .withColumnRenamed("Zone", "DO_zone") \
    .withColumnRenamed("Borough", "DO_borough") \
    .withColumnRenamed("service_zone", "DO_service_zone")




In [12]:
zones_pickup.show(5)

+-------------+-------------+--------------------+---------------+
|PU_LocationID|   PU_borough|             PU_zone|PU_service_zone|
+-------------+-------------+--------------------+---------------+
|            1|          EWR|      Newark Airport|            EWR|
|            2|       Queens|         Jamaica Bay|      Boro Zone|
|            3|        Bronx|Allerton/Pelham G...|      Boro Zone|
|            4|    Manhattan|       Alphabet City|    Yellow Zone|
|            5|Staten Island|       Arden Heights|      Boro Zone|
+-------------+-------------+--------------------+---------------+
only showing top 5 rows


In [14]:

taxi01_zone_df = taxi01_df \
    .join(zones_pickup, taxi01_df.PULocationID == zones_pickup.PU_LocationID, "left") \
    .join(zones_dropoff, taxi01_df.DOLocationID == zones_dropoff.DO_LocationID, "left")


taxi02_zone_df = taxi02_df \
    .join(zones_pickup, taxi02_df.PULocationID == zones_pickup.PU_LocationID, "left") \
    .join(zones_dropoff, taxi02_df.DOLocationID == zones_dropoff.DO_LocationID, "left")

In [15]:
taxi02_zone_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------------+-------------+----------+--------------------+---------------+-------------+----------+--------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_duration_minutes|average_speed_kmph|is_weekend|is_rush_hour|PU_LocationID|PU_borough|             PU_zone|PU_service_zone|DO_LocationID|DO_borough|             DO_zone|DO_service_zone|
+--------+--------------------+---------------------+---------------+-----------

In [ ]:
# Save join data
write_parquet(taxi01_zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\04_join_data\\taxi01")
write_parquet(taxi02_zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\04_join_data\\taxi02")

print("Join data saved successfully!")

Feature engineered data saved successfully!


In [17]:
spark.stop()